In [1]:
import torch

In [2]:
# Model
model=torch.load('eval_gen_FM4.pt')

/tmp/ipykernel_56281/7453637.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model=torch.load('eval_gen_FM4.pt')


In [3]:
import sys
sys.path.insert(0, '/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/src/model/mp20_format')

import torch
import numpy as np
from pymatgen.core import Structure, Lattice

data = torch.load('eval_gen_FM4.pt', map_location='cpu', weights_only=False)

frac_coords = data['frac_coords'].squeeze(0).numpy()
num_atoms   = data['num_atoms'].squeeze(0).numpy()
atom_types  = data['atom_types'].squeeze(0).numpy()
lengths     = data['lengths'].squeeze(0).numpy()
angles      = data['angles'].squeeze(0).numpy()

print(f'Loaded {len(num_atoms)} crystals')

Loaded 40 crystals


In [4]:
structures = []
atom_idx = 0
for i in range(len(num_atoms)):
    n = int(num_atoms[i])
    lattice = Lattice.from_parameters(
        a=lengths[i,0], b=lengths[i,1], c=lengths[i,2],
        alpha=angles[i,0], beta=angles[i,1], gamma=angles[i,2]
    )
    try:
        s = Structure(
            lattice=lattice,
            species=atom_types[atom_idx:atom_idx+n],
            coords=frac_coords[atom_idx:atom_idx+n],
            coords_are_cartesian=False
        )
        structures.append(s)
    except Exception as e:
        import nglview
        nglview.enable_notebook()
        print(f'Crystal {i} failed: {e}')
        structures.append(None)
        atom_idx += n

print(f'{sum(s is not None for s in structures)} valid structures')

40 valid structures


In [5]:
import nglview as nv
import ipywidgets as widgets
from IPython.display import display
import tempfile, os

def view_crystal(idx):
    s = structures[idx]
    if s is None:
        print('Invalid structure')
        return
    
    with tempfile.NamedTemporaryFile(suffix='.cif', delete=False, mode='w') as f:
        tmpfile = f.name
    s.to(filename=tmpfile)
    
    v = nv.show_file(tmpfile, ext='cif')
    v.add_unitcell()
    v.add_ball_and_stick()
    
    print(f'Crystal {idx}: {s.formula} | {len(s)} atoms | vol={s.volume:.1f} Å³')
    display(v)
    os.unlink(tmpfile)

widgets.interact(view_crystal, idx=widgets.IntSlider(min=0, max=len(structures)-1, description='Crystal:'))

interactive(children=(IntSlider(value=0, description='Crystal:', max=39), Output()), _dom_classes=('widget-int…

<function __main__.view_crystal(idx)>